# **YOLO Instance Segmentation**

# 1.환경준비

## (1) 라이브러리 설치

In [ ]:
!pip install ultralytics -q

## (2) 라이브러리 불러오기

In [ ]:
from ultralytics import settings, YOLO
import matplotlib.pyplot as plt
import cv2
import os
from IPython.display import Video
import numpy as np

## (3) YOLO 설정

* 파일 경로 설정

In [ ]:
# 현재 세팅을 확인해 봅시다.
settings

In [ ]:
# 콜랩 파일 탭에 보이는 경로('/content/')로 변경해 봅시다.
settings['datasets_dir'] = '/content/'
settings.update()
settings

# 2.모델 사용

## (1) 모델 다운로드

- 모델의 구조와 해당 구조에 맞게 사전 학습된 가중치를 불러온다.
- Parameters
    * model : 모델 구조 또는 모델 구조 + 가중치 설정. task와 맞는 모델을 선택해야 한다.
    * task : detect, segment, classify, pose 중 택일

In [ ]:
model = YOLO(model='yolo11s-seg.pt')

## (2) 모델 사용 : 이미지

![](https://file.fgtv.com/nanum_files/USERUPLOAD/N2_511/151021160645_01.jpg)

In [ ]:
image_path = 'https://file.fgtv.com/nanum_files/USERUPLOAD/N2_511/151021160645_01.jpg'
results = model.predict(image_path, save=True)
results[0].show()  # 탐지된 객체 출력

In [ ]:
r = results[0]
print(type(r.masks), None if r.masks is None else r.masks.data.shape)
# 예: (N, H, W) 형태면 N개의 개별 인스턴스 마스크가 나온 것
# 개수 확인
print("num instances:", 0 if r.masks is None else r.masks.data.shape[0])

## (3) 세그멘테이션 결과 열어보기

In [ ]:
print(type(results))
print(len(results))

In [ ]:
type(results[0])

In [ ]:
r.boxes

In [ ]:
for box in r.boxes:
    x_min, y_min, x_max, y_max = box.xyxy[0]  # 좌표
    conf = box.conf[0]
    cn = r.names[int(box.cls[0])]  # 클래스 이름
    print(f"좌표: {x_min}, {y_min}, {x_max}, {y_max} | conf. : {conf} | class : {cn}")

* masks
    * 객체별 마스크 정보 (픽셀 단위)를 담고 있는 객체

In [ ]:
results[0].masks

| 속성 | 설명 |
|----|----|
|masks.data | (num_instances, H, W) 형태의 마스크 텐서 (0~1 값)|
|masks.xy | 각 인스턴스의 마스크 외곽선을 polygon으로 표현한 좌표 리스트|
|masks.shape | 마스크 텐서의 크기 (인스턴스 개수, 높이, 너비)|
|masks.bool() | True/False 형태로 변환한 binary mask|


In [ ]:
# 마스크 정보
masks = r.masks

print(masks.data.shape)  # 예: torch.Size([3, 448, 640]) → 3개의 인스턴스
print(masks.data[0])     # 첫 번째 객체의 마스크 (0~1 값의 텐서)

In [ ]:
masks = r.masks

for i in range(3):
    mask = masks.data[i].numpy()
    plt.imshow(mask, cmap='gray')
    plt.show()

## (4) 실습
* 다양한 사진을 찾아서 세그멘테이션 해 봅시다.

## (5) 모델사용 : 동영상

* sample.mp4 파일을 업로드 합니다.

In [ ]:
# colab 파일 업로드
from google.colab import files
uploaded = files.upload()

* 동영상  실행

In [ ]:
# 동영상  실행 및 결과 저장
results = model.predict("sample.mp4", save=True)  # 결과 자동 저장

* 콜랩에서 영상 play를 위한 세팅

In [ ]:
# 영상 코덱 설치
!apt-get install -y ffmpeg

In [ ]:
# AVI to MP4로 변환 (YOLO 탐지 결과 파일명에 맞게 수정)
input_video_path = "runs/segment/predict/sample.avi"  # YOLO 탐지 결과 파일명
output_video_path = "runs/segment/predict/video_converted.mp4"

# FFmpeg를 사용하여 변환 (코덱: libx264)
!ffmpeg -i {input_video_path} -vcodec libx264 {output_video_path}

* 영상 paly

In [ ]:
# YOLO 탐지 결과 동영상 재생
Video("runs/segment/predict/video_converted.mp4", embed=True)